##### Copyright 2025 Google LLC.
Licensed under the Apache 2.0 License.

In [ ]:
# @title Licensed under the Apache 2.0 License (the "License"); { display-mode: "form" }
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 为多模态 LLM输入执行数据适配器

训练一个使用非文本数据的LLM的方法之一是将数据输入LLM's
上下文窗口通过适配器。当您将自然语言文本输入 LLM 时，
首先，它被转换为符号，然后是符号嵌入。每个符号
嵌入是大小为 D 的一维变压器。这导致输入 K
将输入 LLM 的反向构件按 D 矩阵映射到 K。

有鉴于此，将非文本数据注入输入流的简单方法
LLM是使用适配器将非文本数据映射成一组大小的矢量。
D,代表虚拟符号。例如，让我们假设我们有一个矩阵
维度数据集10,由100表示，它代表了与一个维度相关的一些特性。
单输入 LLM (一个单一样本)。我们把 10x100 矩阵输入到一个
适配器，将它映射到一个按 D 矩阵的 T,其中 T是一些虚拟数
符号。然后我们简单地将它预置到输入矩阵中，生成一个(K+T)由 D 生成(K+T)
矩阵，是输入 LLM.

为了培训和评价PROs,在PH-LLM文件中，每个输入样本都有一个
大小数据输入40,代表20的集合值和
20 差异值 20 传感器，在这种情况下，我们学到了 MLP 适配器
(以下执行) 将长度的 1D 的角数 40 映射到 10 由 128 绘制
虚拟符牌嵌入的维基矩阵。然后，我们调制虚拟符牌
LLM 输入矩阵的符号矩阵。

在这里，我们提供一个简单的执行 适配器层在麻松，可以
用于生成虚拟符号。

## 安装相关软件包

In [ ]:
!pip install numpy
!pip install flax
!pip install jax

In [ ]:
from flax import linen as nn
import jax
from jax import numpy as jnp
import numpy as np

## 定义常数

In [ ]:
# LLM变压器中使用的符号嵌入器大小。
MODEL_DIM = 128

# 适配器输入数据的尺寸。
DATA_INPUT_DIM = 40

# 适配器中使用的虚拟符号数。
NUM_VIRTUAL_TOKENS = 10

# MLP 数据适配器中使用的层数。
NUM_MLP_LAYERS = 5

# MLP 隐藏的维度。
MLP_HIDDEN_DIMS = 128

# 训练时使用的批量尺寸。
BATCH_SIZE = 8

# 随机种子
RANDOM_SEED = 123

In [ ]:
class SimpleAdapter(nn.Module):
  """A simple adapter layer for vector input data."""

  # 输入数据的维度。
  input_dims: int = 1
  # 隐藏层的维度。
  hidden_dims: int = 128
  # 符号嵌入的维度。
  token_embedding_dim: int = 128
  # 要分配到数据编码的虚拟符号数。
  num_virtual_tokens: int = 10
  # 适配器中使用的MLP的层数。
  num_layers: int = 5

  @nn.compact
  def __call__(self, input: jnp.ndarray):
    batch_size, _ = input.shape
    x = input
    # 除了最后一层外，所有层都会有激活。
    for i in range(self.num_layers - 1):
      x = nn.Dense(features=self.input_dims)(x)
      x = nn.relu(x)
    # 最后一层映射到想要的输出维度。
    x = nn.Dense(features=self.token_embedding_dim * self.num_virtual_tokens)(x)
    return jnp.reshape(
        x, [batch_size, self.num_virtual_tokens, self.token_embedding_dim]
    )

## 用合成数据测试层。

In [ ]:
g_adapter_layer = SimpleAdapter(
    name='adapter_config',
    input_dims=DATA_INPUT_DIM,
    hidden_dims=MLP_HIDDEN_DIMS,
    token_embedding_dim=MODEL_DIM,
    num_virtual_tokens=NUM_VIRTUAL_TOKENS,
    num_layers=NUM_MLP_LAYERS,
)

In [ ]:
# 我们创建了一个大小的输入矩阵 BATCH_SIZE 由DATA_INPUT_DIM.
# 例如，用默认常数表示
# a 8样本的矩阵，其中每个样本都有大小的数据输入矢量40.
_inputs_np = np.random.normal(
    size=(
        BATCH_SIZE,
        DATA_INPUT_DIM,
    )
)
_inputs = jnp.asarray(_inputs_np)

print(f'We expect {BATCH_SIZE} samples by {DATA_INPUT_DIM}')
print(f'_inputs: {_inputs.shape}')
print('')


_prng_key = jax.random.PRNGKey(seed=RANDOM_SEED)
_initial_vars = g_adapter_layer.init({'params': _prng_key}, _inputs)
_outputs = g_adapter_layer.apply(
    {'params': _initial_vars['params']}, _inputs
)


# 在通过适配器推进输入矩阵后，我们期望
# 适配器的输出应为 BATCH_SIZE 由 NUM_VIRTUAL_TOKENS
# 由 MODEL_DIM 或符号嵌入大小。
print(
    f'We expect {BATCH_SIZE} samples by {NUM_VIRTUAL_TOKENS} tokens by'
    f' {MODEL_DIM} dim.'
)
print(f'_outputs: {_outputs.shape}')

## 下一个去哪里？

适配器层的输出可与输入LLMs的配合。
如何做到这一点的细节将取决于具体的执行情况。